In [11]:
import pandas as pd
from transformers import pipeline

DATA_PATH = "tiktok.csv"
TEXT_COLUMN = "Comment"

# Load and cleaning dataset
df = pd.read_csv(DATA_PATH)
print(df.info())
df = df.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)
print(f"Loaded {len(df)} comments")
print(df.head())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 910 entries, 0 to 909
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   User     910 non-null    object
 1   Comment  885 non-null    object
 2   Date     910 non-null    object
 3   Reply    910 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 28.6+ KB
None
Loaded 885 comments
                                    User  \
0  https://www.tiktok.com/@rubegoldburgh   
1     https://www.tiktok.com/@bmaperkins   
2   https://www.tiktok.com/@john.boy2836   
3      https://www.tiktok.com/@anera5703   
4      https://www.tiktok.com/@ssviperfl   

                                             Comment    Date  Reply  
0    Why the fuck is the mayo in the ketchup bottle?  Jun-15   7357  
1  For the love of God, leave the egg off the burger  Jun-15   1655  
2                                        It’s giving  Jul-14    164  
3                                 Worse burger ever. 

In [ ]:
# Categories aspect want to know
ASPECTS = {
    "taste":       ["taste", "flavor", "delicious", "gross", "yummy"],
    "price":       ["price", "expensive", "cheap", "cost"],
    "appearance":  ["look", "looks", "presentation", "ugly", "gross looking"],
    "ingredients": ["egg", "mayo", "ketchup", "cheese", "meat", "bun"],
}

In [ ]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

In [14]:
# Extract aspect mentions
def get_aspect_sentiments(text: str):
    text_lower = text.lower()
    found = []
    for aspect, keywords in ASPECTS.items():
        if any(kw in text_lower for kw in keywords):
            result = sentiment_model(text[:512])[0]  # truncate to avoid token limit
            found.append({"aspect": aspect, "label": result["label"], "score": round(result["score"], 3)})
    return found

In [18]:
# Run pipeline
records = []
for _, row in df.iterrows():
    for r in get_aspect_sentiments(row[TEXT_COLUMN]):
        records.append({"comment": row[TEXT_COLUMN], **r})

absa_df = pd.DataFrame(records)
absa_df.tail()

,comment,aspect,label,score
346,You better bring back the mayo bottle and sque...,ingredients,neutral,0.537
347,"You cracked an egg there, buddy. You’re not ma...",ingredients,negative,0.863
348,you know its going to be expensive when the ch...,price,negative,0.554
349,You lost me at the toxic-chemical American cheese,ingredients,negative,0.946
350,yummy,taste,positive,0.448


## 6. Save and review results

In [6]:
# Save results
absa_df.to_csv("absa_results.csv", index=False)

print("Sentiment distribution per aspect:")
absa_df.groupby(["aspect", "label"]).size().unstack(fill_value=0)

Sentiment distribution per aspect:


label,negative,neutral,positive
aspect,,,
appearance,23,4,45
ingredients,143,66,35
price,4,1,0
taste,12,0,18
